In [ ]:
#### setting dictionaries - mappings 

In [ ]:
### chmi weather stations - data processing

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [ ]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open("data/wsi_dict.csv","w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [ ]:
### chmi weather variables

### filtering only needed ones

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

In [ ]:
### mapping variables names

df_chmi['ELEMENT_NAME'] = df_chmi['ELEMENT'].map(chmi_vars_dict)

In [ ]:
### golemio air quality 

#### air quality metadata processing

station_cols = {
    'geometry.coordinates': 'coordinates', 
    'properties.id': 'id', 
    'properties.name': 'name', 
    'properties.district': 'district', 
    'properties.measurement.components.type': 'components'
}

air_quality_stations = df[station_cols.keys()]

air_quality_stations = air_quality_stations.rename(columns=station_cols)

air_quality_stations = (
    air_quality_stations.groupby('id', as_index=False)
    .agg({
        'coordinates': 'first',
        'name': 'first',
        'district': 'first',
        'components': list
    })
)

air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")


In [ ]:

#air quality stations dictionary

air_stat_dict = dict(
    zip(
        air_quality_stations['id'].astype(str),
        air_quality_stations['name'].astype(str)  
    )
)